# RoadScan — prepare small evaluation inputs on CPU

The first GPU evaluation was cancelled after prolonged initialization with no visible logs. CPU checks successfully loaded both checkpoints and development images. This recovery stage copies only the two unchanged best checkpoints, fixed development manifests and training diagnostics into a small output package. It does not install models, train a model, select a new checkpoint or read test annotations.

Run on CPU, then use Quick Save with Save Output enabled. The next notebook evaluates these saved files at 640, 768 and 960. SHA256 hashes verify that model contents are unchanged.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import hashlib
import json

SOURCE = Path('/kaggle/input/notebooks/rimasabunasser/06-roadscan-quality-improvement/roadscan_v2')
OUT = Path('/kaggle/working/resolution_inputs')
assert (SOURCE / 'phase2_checkpoint.txt').is_file()
(OUT / 'models').mkdir(parents=True, exist_ok=True)
(OUT / 'histories').mkdir(exist_ok=True)
for name in ['yolo26s_768', 'yolo26s_960_finetune']:
    print('Copying saved checkpoint:', name, flush=True)
    shutil.copy2(SOURCE / 'runs' / name / 'weights/best.pt', OUT / 'models' / (name + '.pt'))
    shutil.copy2(SOURCE / 'runs' / name / 'results.csv', OUT / 'histories' / (name + '.csv'))
for name in ['in_domain_dev_manifest.csv', 'china_dev_manifest.csv', 'development_manifest.csv',
             'training_gap_diagnostic.json', 'training_curves.png']:
    shutil.copy2(SOURCE / name, OUT / name)
files = sorted(p for p in OUT.rglob('*') if p.is_file())
checksums = {str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(OUT / 'SHA256.json').write_text(json.dumps(checksums, indent=2))
(OUT / 'PROVENANCE.json').write_text(json.dumps(dict(
    source_notebook='rimasabunasser/06-roadscan-quality-improvement',
    original_run_version=359127115, created_utc=datetime.now(timezone.utc).isoformat(),
    purpose='Evaluation inputs only: two unchanged checkpoints, development manifests and training diagnostics.',
    training_performed=False, test_data_loaded=False), indent=2))
print('INPUT PACKAGE READY:', OUT, flush=True)
print('Total size (MiB):', round(sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 2**20, 2), flush=True)
print('Files:', [str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()], flush=True)


Copying saved checkpoint: yolo26s_768
Copying saved checkpoint: yolo26s_960_finetune
INPUT PACKAGE READY: /kaggle/working/resolution_inputs
Total size (MiB): 39.91
Files: ['training_curves.png', 'china_dev_manifest.csv', 'PROVENANCE.json', 'in_domain_dev_manifest.csv', 'development_manifest.csv', 'training_gap_diagnostic.json', 'SHA256.json', 'histories/yolo26s_960_finetune.csv', 'histories/yolo26s_768.csv', 'models/yolo26s_768.pt', 'models/yolo26s_960_finetune.pt']
